## Section 0 — Setup

In [11]:
import os, gc, json, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

import wandb

WANDB_PROJECT = "24f2003563-t22026"

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [12]:
train = pd.read_csv("/content/train.csv")
test  = pd.read_csv("/content/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()

Train shape: (2000, 8)
Test shape: (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [13]:
def build_dataset(df, has_answer=True):
    rows = []
    for _, row in df.iterrows():
        for option in ['A', 'B', 'C', 'D', 'E']:
            text = f"Question: {row['prompt']} Candidate Answer: {row[option]}"
            item = {
                "id": row['id'],
                "prompt": row['prompt'],
                "option_text": row[option],
                "text": text,
                "option": option
            }
            if has_answer:
                item["label"] = 1 if option == row['answer'] else 0
            rows.append(item)
    return pd.DataFrame(rows)

train_processed = build_dataset(train, has_answer=True)
test_processed  = build_dataset(test,  has_answer=False)

print(train_processed.shape, test_processed.shape)
train_processed.head(10)

(10000, 6) (2500, 5)


,id,prompt,option_text,text,option,label
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Question: Pick the best possible answer: What ...,A,0
1,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans do not e...,Question: Pick the best possible answer: What ...,B,1
2,1,Pick the best possible answer: What is Martin ...,Martin Heidegger does not believe in the exist...,Question: Pick the best possible answer: What ...,C,0
3,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that the relationshi...,Question: Pick the best possible answer: What ...,D,0
4,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that time is an illu...,Question: Pick the best possible answer: What ...,E,0
5,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Question: What is accelerator-based light-ion ...,A,1
6,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Question: What is accelerator-based light-ion ...,B,0
7,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Question: What is accelerator-based light-ion ...,C,0
8,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Question: What is accelerator-based light-ion ...,D,0
9,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Question: What is accelerator-based light-ion ...,E,0


In [14]:
# Train / Val split
train_ids, val_ids = train_test_split(train['id'].unique(), test_size=0.15, random_state=42)

train_df = train_processed[train_processed['id'].isin(train_ids)].reset_index(drop=True)
val_df   = train_processed[train_processed['id'].isin(val_ids)].reset_index(drop=True)

train_q = train[train['id'].isin(train_ids)].reset_index(drop=True)
val_q   = train[train['id'].isin(val_ids)].reset_index(drop=True)

print(f"Train questions: {len(train_ids)} | Val questions: {len(val_ids)}")


Train questions: 1700 | Val questions: 300


In [15]:
# MAP@3 metric
def map_at_3(true_answers, pred_lists):
    """
    true_answers: list of correct option letters, e.g. ['A','C',...]
    pred_lists:   list of lists of top-3 predicted letters, e.g. [['A','B','C'], ...]
    """
    scores = []
    for true, preds in zip(true_answers, pred_lists):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return float(np.mean(scores))

def scores_to_submission(id_list, option_list, score_list, out_path=None):
    """Convert per-option scores into a ranked top-3 submission dataframe."""
    df = pd.DataFrame({"id": id_list, "option": option_list, "score": score_list})
    rows = []
    for qid, g in df.groupby("id"):
        ranked = g.sort_values("score", ascending=False)
        top3 = ranked["option"].values[:3].tolist()
        rows.append({"ID": qid, "Prediction": " ".join(top3)})
    sub = pd.DataFrame(rows)
    if out_path:
        sub.to_csv(out_path, index=False)
    return sub


## TF-IDF / Word2Vec Baseline + Cosine Similarity


In [16]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

train_df["text_clean"] = train_df["text"].apply(clean_text)
val_df["text_clean"]   = val_df["text"].apply(clean_text)
test_processed["text_clean"] = test_processed["text"].apply(clean_text)

In [17]:
# TF-IDF vectorizer
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(train_df["text_clean"])
X_val_tfidf   = tfidf.transform(val_df["text_clean"])
X_test_tfidf  = tfidf.transform(test_processed["text_clean"])

prompt_vec_train = tfidf.transform(train_df["prompt"].apply(clean_text))
option_vec_train = tfidf.transform(train_df["option_text"].apply(clean_text))
cos_sim_train = np.array([
    cosine_similarity(prompt_vec_train[i], option_vec_train[i])[0, 0]
    for i in range(prompt_vec_train.shape[0])
])

prompt_vec_val = tfidf.transform(val_df["prompt"].apply(clean_text))
option_vec_val = tfidf.transform(val_df["option_text"].apply(clean_text))
cos_sim_val = np.array([
    cosine_similarity(prompt_vec_val[i], option_vec_val[i])[0, 0]
    for i in range(prompt_vec_val.shape[0])
])

prompt_vec_test = tfidf.transform(test_processed["prompt"].apply(clean_text))
option_vec_test = tfidf.transform(test_processed["option_text"].apply(clean_text))
cos_sim_test = np.array([
    cosine_similarity(prompt_vec_test[i], option_vec_test[i])[0, 0]
    for i in range(prompt_vec_test.shape[0])
])

print("Cosine sim feature ready. Example values:", cos_sim_train[:5])


Cosine sim feature ready. Example values: [0.1207969  0.13326118 0.2848606  0.34545644 0.10765669]


In [18]:
from scipy.sparse import hstack, csr_matrix

X_train_full = hstack([X_train_tfidf, csr_matrix(cos_sim_train).T])
X_val_full   = hstack([X_val_tfidf,   csr_matrix(cos_sim_val).T])
X_test_full  = hstack([X_test_tfidf,  csr_matrix(cos_sim_test).T])

y_train = train_df["label"].values
y_val   = val_df["label"].values

logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg.fit(X_train_full, y_train)

val_probs_tfidf = logreg.predict_proba(X_val_full)[:, 1]
val_preds_tfidf = (val_probs_tfidf > 0.5).astype(int)

acc_tfidf = accuracy_score(y_val, val_preds_tfidf)
f1_tfidf  = f1_score(y_val, val_preds_tfidf)
print(f"TF-IDF + LogReg — Val Accuracy: {acc_tfidf:.4f} | F1: {f1_tfidf:.4f}")


TF-IDF + LogReg — Val Accuracy: 0.9480 | F1: 0.8825


In [19]:
# Compute MAP@3 for TF-IDF model
val_sub_tfidf = scores_to_submission(val_df["id"].values, val_df["option"].values, val_probs_tfidf)
val_answers = val_q.set_index("id").loc[val_sub_tfidf["ID"], "answer"].values
pred_lists_tfidf = val_sub_tfidf["Prediction"].str.split().tolist()

map3_tfidf = map_at_3(val_answers, pred_lists_tfidf)
print(f"TF-IDF + LogReg — Val MAP@3: {map3_tfidf:.4f}")


TF-IDF + LogReg — Val MAP@3: 0.9917
